# Open Quantum: a small hardware observation study

This notebook runs **on your computer**, after the simulation pilot. It prepares 18 entries: Bell, GHZ, and Bernstein–Vazirani at 256 and 1,024 shots, each repeated three times. It targets Rigetti Cepheus-1-108Q only. If that device is unavailable, postpone the study. The public PhasorQ website never reads credentials or submits jobs.

Install the optional dependency with `uv sync --group dev --extra hardware`. Set `OPENQUANTUM_CLIENT_ID` and `OPENQUANTUM_CLIENT_SECRET` in your local environment, plus `OPENQUANTUM_ORGANIZATION_ID` if you belong to multiple organizations. Do not write them into the notebook. Work from the repository root.

Public Compute circuits are not confidential. Use these public teaching examples only, and attribute Quantum Rings/Open Quantum when publishing results. The proposed ceiling is 5 Spark credits, currently described as about $10 equivalent. Exact live quotes and the Spark balance decide whether a job can proceed. No purchased Full credits are used.

In [ ]:
from pathlib import Path
from src.open_quantum_study import (
    initialize_study, read_manifest, verify_target, prepare, submit_prepared,
    attach_job_id, collect,
)
study_dir = Path('study_private/open_quantum')
manifest_path = study_dir / 'manifest.json'
if not manifest_path.exists():
    initialize_study(study_dir)
manifest = read_manifest(manifest_path)
print(len(manifest['jobs']), 'planned entries')
print([(item['entry_id'], item['status'], item['job_id']) for item in manifest['jobs']])

## Verify and prepare one entry
`verify_target()` checks the live Rigetti backend and whether Qiskit can compile our three circuits. `prepare()` uploads one public example and asks Open Quantum for a quote. It does not create a hardware job. Review the quote and your available Spark credits before running the submission cell.

In [ ]:
entry_id = 'bell-256-r1'
print(verify_target())
print(prepare(manifest_path, entry_id))

## Submit only after reviewing the live quote
The function checks that the Public/Standard quote fits both the study ceiling and the Spark balance. It then requires you to type the entry-specific confirmation before creating a job. A `submission_pending` entry is deliberately **never retried automatically**. If execution is interrupted after confirmation, find the job in the Open Quantum portal and attach its ID with the recovery cell.

In [ ]:
# Creates one real hardware job only after an exact typed confirmation.
job_id = submit_prepared(manifest_path, entry_id)
print('Saved job ID:', job_id)

In [ ]:
# Recovery only: use the actual job ID visible in the provider portal.
# attach_job_id(manifest_path, entry_id, 'actual-provider-job-id')

## Collect a completed job
Run this later if the job is still queued. A completed job produces a PhasorQ JSON record in `study_private/open_quantum/records/`; import that file in the website. Raw provider output is kept privately too. If the provider changes its result shape, the importer stops instead of guessing measurement order or counts. Missing calibration and charge remain unknown. Repeat the prepare, submit, and collect steps for the other entries as your live quote and Spark balance permit.

In [ ]:
record_path = collect(manifest_path, entry_id)
print(record_path or 'Job has not completed yet')

Report actual completed jobs, the backend and dates, provider attribution, missing calibration, and any runs that could not be imported. This small observational study does not establish an optimization advantage or validate the synthetic noise model.